# 🌊 Week 4, Day 5 — June 12, 2026
## Habitat-Type Segmentation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw", "processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata", "visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026

In [ ]:
master_v2 = pd.read_csv(DIRS['processed']+'/master_table_v2.csv')
df_s      = pd.read_csv(DIRS['processed']+'/species_clean_final.csv')
print(f'master_v2: {master_v2.shape}, species: {df_s.shape}')

## Step 1: Derive Habitat_Type from Coordinates

The `india_cms` dataset has a `location_type` column but it's mostly `Terrestrial` (not useful for marine habitat). We'll derive habitat from coordinate rules based on oceanographic knowledge of the Indian Ocean.

In [ ]:
def assign_habitat(lat, lon):
    """
    Rule-based habitat assignment for North Indian Ocean.
    
    Coral Reef zones:
      - Gulf of Mannar (8-12°N, 76-80°E) — one of India's richest reef systems
      - Lakshadweep (8-15°N, 72-74°E) — atoll reef chain
      - Andaman & Nicobar (10-14°N, 92-95°E) — diverse coral ecosystem
    
    Mangrove zones:
      - Sundarbans (21-23°N, 87-90°E) — world's largest mangrove
      - Mumbai coast (19-22°N, 72-73°E) — urban mangroves
      - Andhra/Odisha coast (13-16°N, 80-82°E) — coastal mangroves
    
    Open Ocean: everything else
    """
    # Coral Reef
    if  (8  <= lat <= 12 and 76 <= lon <= 80): return 'Coral_Reef'
    elif (8  <= lat <= 15 and 72 <= lon <= 74): return 'Coral_Reef'
    elif (10 <= lat <= 14 and 92 <= lon <= 95): return 'Coral_Reef'
    # Mangrove
    elif (21 <= lat <= 23 and 87 <= lon <= 90): return 'Mangrove'
    elif (19 <= lat <= 22 and 72 <= lon <= 73): return 'Mangrove'
    elif (13 <= lat <= 16 and 80 <= lon <= 82): return 'Mangrove'
    # Default
    else: return 'Open_Ocean'

df_s['grid_lat'] = np.floor(df_s['latitude']).astype(int)
df_s['grid_lon'] = np.floor(df_s['longitude']).astype(int)
df_s['Habitat_Type'] = df_s.apply(lambda r: assign_habitat(r['grid_lat'], r['grid_lon']), axis=1)

print('Habitat assignment complete:')
print(df_s['Habitat_Type'].value_counts())
print()
pct = df_s['Habitat_Type'].value_counts(normalize=True).round(3)*100
print('Percentage split:')
for h, p in pct.items():
    print(f'  {h:<15}: {p:.1f}%')

## Step 2: Species Trends by Habitat

In [ ]:
# Species count by habitat and year
habitat_yearly = (
    df_s.groupby(['Habitat_Type','year'])
    ['species_common_name'].count()
    .reset_index()
    .rename(columns={'species_common_name':'Species_Count'})
)

print('Species observations by Habitat × Year:')
pivot = habitat_yearly.pivot(index='year', columns='Habitat_Type', values='Species_Count').fillna(0).astype(int)
print(pivot.to_string())

## Step 3: Plastic Density by Habitat Zone

In [ ]:
# Apply same habitat function to master table
master_v2['Habitat_Type'] = master_v2.apply(
    lambda r: assign_habitat(r['grid_lat'], r['grid_lon']), axis=1
)

plastic_by_habitat = (
    master_v2.dropna(subset=['Plastic_Density_kg_km2'])
    .groupby('Habitat_Type')['Plastic_Density_kg_km2']
    .agg(['sum','mean','count'])
    .reset_index()
    .rename(columns={'sum':'Total_Density','mean':'Avg_Density','count':'Grid_Cells'})
)
plastic_by_habitat['Contribution_Pct'] = (
    plastic_by_habitat['Total_Density'] / plastic_by_habitat['Total_Density'].sum() * 100
).round(1)

print('Plastic density by habitat zone:')
print(plastic_by_habitat.round(4).to_string(index=False))

## Step 4: 3-Panel Habitat Comparison Plot

In [ ]:
habitat_colors = {
    'Coral_Reef':  '#e74c3c',
    'Mangrove':    '#27ae60',
    'Open_Ocean':  '#3498db'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
sns.set_style('whitegrid')

# ── Panel 1: Species trend by habitat ──
ax1 = axes[0]
for habitat, color in habitat_colors.items():
    subset = habitat_yearly[habitat_yearly['Habitat_Type']==habitat]
    ax1.plot(subset['year'], subset['Species_Count'],
             color=color, linewidth=2.5, marker='o', markersize=6,
             label=habitat.replace('_',' '))
    ax1.fill_between(subset['year'], subset['Species_Count'], alpha=0.08, color=color)
ax1.set_title('Species Observations\nby Habitat Type (2015–2025)', fontweight='bold')
ax1.set_xlabel('Year')
ax1.set_ylabel('Observation Count (log scale)')
ax1.set_yscale('log')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Panel 2: Plastic density by habitat (bar) ──
ax2 = axes[1]
bars = ax2.bar(
    plastic_by_habitat['Habitat_Type'].str.replace('_',' '),
    plastic_by_habitat['Total_Density'],
    color=[habitat_colors.get(h,'gray') for h in plastic_by_habitat['Habitat_Type']],
    edgecolor='black', linewidth=0.8, alpha=0.85
)
ax2.set_title('Total Plastic Density\nby Habitat Zone', fontweight='bold')
ax2.set_ylabel('Total Plastic Density (kg/km²)')
for bar, row in zip(bars, plastic_by_habitat.itertuples()):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0001,
             f'{row.Contribution_Pct:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# ── Panel 3: Species richness heatmap by habitat and year ──
ax3 = axes[2]
pivot_norm = pivot.copy().astype(float)
for col in pivot_norm.columns:
    pivot_norm[col] = pivot_norm[col] / pivot_norm[col].max()  # normalize 0-1 per habitat

sns.heatmap(pivot_norm.T, ax=ax3, cmap='YlOrRd',
            annot=pivot.T.values, fmt='d',
            cbar_kws={'label': 'Normalised Intensity'},
            linewidths=0.5, linecolor='white')
ax3.set_title('Species Count Heatmap\n(by Habitat × Year, raw count annotated)', fontweight='bold')
ax3.set_xlabel('Year')
ax3.set_ylabel('Habitat Type')

plt.suptitle('June 12, 2026 — Habitat-Type Segmentation\nCoral Reef vs Mangrove vs Open Ocean Ecological Impact',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week4_Day5_habitat_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Habitat segmentation plot saved ✅')

## Step 5: Localized Impact Findings

In [ ]:
print('HABITAT-LEVEL ECOLOGICAL IMPACT ANALYSIS')
print('='*55)
print()
print('CORAL REEF ZONES (Gulf of Mannar, Lakshadweep, Andaman):')
cr = habitat_yearly[habitat_yearly['Habitat_Type']=='Coral_Reef']
cr_plastic = plastic_by_habitat[plastic_by_habitat['Habitat_Type']=='Coral_Reef']
print(f'  Species records: {cr["Species_Count"].sum():,}  ({cr["Species_Count"].sum()/len(df_s)*100:.1f}% of total)')
if len(cr_plastic) > 0:
    print(f'  Plastic density: {cr_plastic["Contribution_Pct"].values[0]:.1f}% of total')
print('  ⚠️  Coral bleaching risk: HIGH (SST >28.5°C + plastic entanglement)')
print()
print('MANGROVE ZONES (Sundarbans, Mumbai coast, Andhra coast):')
mg = habitat_yearly[habitat_yearly['Habitat_Type']=='Mangrove']
mg_plastic = plastic_by_habitat[plastic_by_habitat['Habitat_Type']=='Mangrove']
print(f'  Species records: {mg["Species_Count"].sum():,}  ({mg["Species_Count"].sum()/len(df_s)*100:.1f}% of total)')
if len(mg_plastic) > 0:
    print(f'  Plastic density: {mg_plastic["Contribution_Pct"].values[0]:.1f}% of total')
print('  ⚠️  Filter feeders (crabs, molluscs) most at risk from microplastic ingestion')
print()
print('OPEN OCEAN:')
oo = habitat_yearly[habitat_yearly['Habitat_Type']=='Open_Ocean']
oo_plastic = plastic_by_habitat[plastic_by_habitat['Habitat_Type']=='Open_Ocean']
print(f'  Species records: {oo["Species_Count"].sum():,}  ({oo["Species_Count"].sum()/len(df_s)*100:.1f}% of total)')
if len(oo_plastic) > 0:
    print(f'  Plastic density: {oo_plastic["Contribution_Pct"].values[0]:.1f}% of total')
print('  Pelagic species (dolphins, whales, turtles) face entanglement + ingestion risk')

In [ ]:
# Save master with Habitat_Type
master_v2.to_csv(DIRS['processed']+'/master_table_v2.csv', index=False)
df_s.to_csv(DIRS['processed']+'/species_with_habitat.csv', index=False)
print('Saved master_table_v2.csv and species_with_habitat.csv ✅')